# IR Assignment 2 — Step 2: Field-Aware BM25F Retrieval

**Research question:** Does preserving patent document structure (separate title, abstract, claims, description fields) and weighting them independently give better, more believable rankings than the plain BM25 baseline from Step 1?

This notebook:

1. Loads the parsed patent dataframe from Step 1 (no re-parsing)
2. Audits field coverage (title, abstract, claims, description)
3. Tokenises each field separately instead of collapsing into one blob
4. Builds a field-aware BM25F retrieval pipeline using per-field BM25 indexes
5. Runs sanity-check queries and compares BM25 vs BM25F side-by-side
6. Shows concrete examples where field-aware scoring changes rankings
7. Saves ranked outputs for downstream notebooks

In [1]:
%pip install -q pandas tqdm rank-bm25


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import re
import json

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

/Users/ledionalame/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 — Control Panel

In [3]:
# project paths — must match Step 1
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
RESULTS_DIR = PROJECT_ROOT / "results" / "bm25f"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# path to Step 1 parsed output
PARSED_PATH = INTERIM_DIR / "parsed_patents.jsonl"

# retrieval settings
TOP_K = 10

# ---------- BM25F field weights ----------
# Rationale:
#   title    — short, high-signal; a match here is very strong evidence
#   abstract — concise summary of the invention; second strongest signal
#   claims   — defines the legal scope; important but verbose
#   description — long background text; term matches are weaker signals
FIELD_WEIGHTS = {
    "title":       3.0,
    "abstract":    2.0,
    "claims":      1.5,
    "description": 1.0,
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PARSED_PATH exists:", PARSED_PATH.exists())
print("RESULTS_DIR:", RESULTS_DIR)
print("Field weights:", FIELD_WEIGHTS)

PROJECT_ROOT: /Users/ledionalame/Documents/QMUL/Second Semester/Information Retrieval/Assignment 2/Project/IR-CW2
PARSED_PATH exists: True
RESULTS_DIR: /Users/ledionalame/Documents/QMUL/Second Semester/Information Retrieval/Assignment 2/Project/IR-CW2/results/bm25f
Field weights: {'title': 3.0, 'abstract': 2.0, 'claims': 1.5, 'description': 1.0}


## 2 — Load Parsed Data from Step 1

**Difference from Step 1:** we do not re-parse XML. We directly load the JSONL that Step 1 produced, keeping the pipeline reproducibly chained.

In [4]:
if not PARSED_PATH.exists():
    raise FileNotFoundError(
        f"Step 1 output not found at {PARSED_PATH}.\n"
        "Run 01_mvp_clefip_bm25.ipynb first to generate parsed_patents.jsonl"
    )

docs_df = pd.read_json(PARSED_PATH, lines=True)

print(f"Loaded {len(docs_df):,} patents from Step 1")
print(f"Columns: {list(docs_df.columns)}")
docs_df.head(3)

Loaded 2,000 patents from Step 1
Columns: ['ucid', 'title', 'abstract', 'claims', 'description', 'ipc', 'source_file']


,ucid,title,abstract,claims,description,ipc,source_file
0,WO-1979000001-A1,APPARATUS FOR DETERMINING THE EPIDERMIC GROUP ...,,REVENDICATIONS 1) Dispositif permettant la déf...,La présente invention constitue un dispositif ...,2 A61B 5/00,/Users/ledionalame/Documents/QMUL/Second Semes...
1,WO-1979000002-A1,IMPROVEMENTS RELATING TO MEMBRANE ELECTROPHORESIS,,CLAIMS 1. An electrophoresis membrane of known...,IMPROVEMENTS RELATING TO MEMBRANE ELECTROPHORE...,2 G01N 27/26 B01D 13/02,/Users/ledionalame/Documents/QMUL/Second Semes...
2,WO-1979000005-A1,IMPROVED CLUTCH-BRAKE SYSTEM FOR ROTARY MOWER,,"WHAT IS CLAIMED IS: "" 1. A mechanism for a rot...",IMPROVED CLUTCH-BRAKE SYSTEM FOR ROTARY MOWER ...,2 F16D 67/02,/Users/ledionalame/Documents/QMUL/Second Semes...


## 3 — Field Coverage Audit

**New in Step 2.** Step 1 never checked how many patents actually have usable text in each field. BM25F is only meaningful if the fields contain real content. This audit tells us what we are working with.

In [5]:
FIELDS = ["title", "abstract", "claims", "description"]

# fill missing fields with empty strings so downstream code never hits NaN
for field in FIELDS:
    if field not in docs_df.columns:
        print(f"WARNING: '{field}' column missing — creating empty column")
        docs_df[field] = ""
    else:
        docs_df[field] = docs_df[field].fillna("").astype(str)

# compute coverage statistics
coverage = {}
for field in FIELDS:
    non_empty = (docs_df[field].str.strip() != "").sum()
    avg_len = docs_df[field].str.split().str.len().mean()
    coverage[field] = {
        "non_empty_count": int(non_empty),
        "coverage_pct": round(100 * non_empty / len(docs_df), 1),
        "avg_word_count": round(avg_len, 1),
    }

coverage_df = pd.DataFrame(coverage).T
coverage_df.index.name = "field"
print("Field coverage summary:")
coverage_df

Field coverage summary:


,non_empty_count,coverage_pct,avg_word_count
field,,,
title,2000.0,100.0,7.8
abstract,1906.0,95.3,107.0
claims,1830.0,91.5,950.7
description,1832.0,91.6,5544.7


This table tells you which fields are worth weighting. If claims or description have very low coverage, their BM25F contribution will naturally be small. Fields with high coverage and reasonable word counts are the ones that will drive ranking differences.

## 4 — Per-Field Tokenisation

**Key difference from Step 1:** Step 1 tokenised a single `search_text = title + abstract` blob. Here we tokenise each field independently, preserving field boundaries. This is the structural foundation of BM25F.

In [6]:
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

def tokenize(text: str):
    """Lowercase alphanumeric tokeniser (same as Step 1 for consistency)."""
    return TOKEN_RE.findall(text.lower())

# tokenise each field separately
for field in FIELDS:
    col_name = f"tokens_{field}"
    docs_df[col_name] = docs_df[field].map(tokenize)
    print(f"{col_name}: avg {docs_df[col_name].str.len().mean():.0f} tokens/doc")

# also rebuild the Step 1 combined tokens for fair comparison
docs_df["search_text"] = (
    docs_df["title"].fillna("") + " " + docs_df["abstract"].fillna("")
).str.replace(r"\s+", " ", regex=True).str.strip()
docs_df["tokens_combined"] = docs_df["search_text"].map(tokenize)

# drop documents with zero tokens across all fields
has_tokens = docs_df[[f"tokens_{f}" for f in FIELDS]].apply(
    lambda row: any(len(t) > 0 for t in row), axis=1
)
docs_df = docs_df[has_tokens].reset_index(drop=True)
print(f"\nDocuments with at least one non-empty field: {len(docs_df):,}")

tokens_title: avg 8 tokens/doc
tokens_abstract: avg 109 tokens/doc
tokens_claims: avg 988 tokens/doc
tokens_description: avg 5825 tokens/doc

Documents with at least one non-empty field: 2,000


## 5 — Build Per-Field BM25 Indexes

**Core architectural change.** Instead of one BM25 index over concatenated text, we build four independent BM25 indexes — one per field. Each index has its own document length statistics, IDF values, and TF saturation. This means a term that appears in every description (low IDF in description) can still be discriminative in titles (high IDF in title).

In [7]:
# build one BM25 index per field
bm25_indexes = {}
for field in FIELDS:
    corpus = docs_df[f"tokens_{field}"].tolist()
    bm25_indexes[field] = BM25Okapi(corpus)
    print(f"Built BM25 index for '{field}': {len(corpus):,} docs")

# also build the baseline combined index (replicates Step 1)
bm25_baseline = BM25Okapi(docs_df["tokens_combined"].tolist())
print(f"\nBuilt baseline BM25 index (title+abstract): {len(docs_df):,} docs")

Built BM25 index for 'title': 2,000 docs
Built BM25 index for 'abstract': 2,000 docs
Built BM25 index for 'claims': 2,000 docs
Built BM25 index for 'description': 2,000 docs

Built baseline BM25 index (title+abstract): 2,000 docs


## 6 — BM25F Search Function

The BM25F scoring computes a weighted sum of per-field BM25 scores:

$$\text{score}_{\text{BM25F}}(q, d) = \sum_{f \in \text{fields}} w_f \cdot \text{BM25}(q, d_f)$$

This is a practical approximation. The canonical BM25F merges field TF values before saturation, but the weighted-sum approach is widely used, easier to implement, and lets us reuse the `rank_bm25` library directly.

In [8]:
def search_bm25f(query_text: str, field_weights: dict = None,
                 top_k: int = 10, exclude_ucid: str = None):
    """Field-aware BM25F search. Scores each field independently then combines with weights."""
    if field_weights is None:
        field_weights = FIELD_WEIGHTS

    query_tokens = tokenize(query_text)
    if len(query_tokens) == 0:
        raise ValueError("Query contains no usable tokens after tokenization.")

    combined_scores = np.zeros(len(docs_df))
    field_score_breakdown = {}

    for field, weight in field_weights.items():
        field_scores = bm25_indexes[field].get_scores(query_tokens)
        field_score_breakdown[field] = field_scores
        combined_scores += weight * field_scores

    result_df = docs_df[["ucid", "title", "abstract", "ipc", "source_file"]].copy()
    result_df["bm25f_score"] = combined_scores

    for field in field_weights:
        result_df[f"score_{field}"] = field_score_breakdown[field]

    if exclude_ucid is not None:
        result_df = result_df[result_df["ucid"] != exclude_ucid]

    result_df = result_df.sort_values("bm25f_score", ascending=False).head(top_k).reset_index(drop=True)
    result_df.insert(0, "rank", range(1, len(result_df) + 1))
    return result_df


def search_bm25_baseline(query_text: str, top_k: int = 10, exclude_ucid: str = None):
    """Plain BM25 on title+abstract (replicates Step 1)."""
    query_tokens = tokenize(query_text)
    if len(query_tokens) == 0:
        raise ValueError("Query contains no usable tokens after tokenization.")

    scores = bm25_baseline.get_scores(query_tokens)
    result_df = docs_df[["ucid", "title", "abstract", "ipc", "source_file"]].copy()
    result_df["bm25_score"] = scores

    if exclude_ucid is not None:
        result_df = result_df[result_df["ucid"] != exclude_ucid]

    result_df = result_df.sort_values("bm25_score", ascending=False).head(top_k).reset_index(drop=True)
    result_df.insert(0, "rank", range(1, len(result_df) + 1))
    return result_df


def search_by_ucid_bm25f(query_ucid: str, top_k: int = 10, field_weights: dict = None):
    """Patent-as-query using BM25F."""
    row = docs_df.loc[docs_df["ucid"] == query_ucid]
    if row.empty:
        raise KeyError(f"ucid not found: {query_ucid}")
    row = row.iloc[0]
    query_text = f"{row['title']} {row['abstract']}".strip()
    return search_bm25f(query_text, field_weights=field_weights,
                        top_k=top_k, exclude_ucid=query_ucid)


print("Search functions defined: search_bm25f, search_bm25_baseline, search_by_ucid_bm25f")


Search functions defined: search_bm25f, search_bm25_baseline, search_by_ucid_bm25f


## 7 — Sanity Check Queries

We run the same queries from Step 1 through both BM25 and BM25F to see if the field-aware approach changes anything.

In [9]:
demo_query = "wireless power transfer charging device"

print("=" * 80)
print(f"QUERY: '{demo_query}'")
print("=" * 80)

# baseline BM25 (Step 1 approach)
baseline_results = search_bm25_baseline(demo_query, top_k=TOP_K)
print("\n--- BM25 Baseline (title+abstract only) ---")
print(baseline_results[["rank", "ucid", "bm25_score", "title"]].to_string(index=False))

# BM25F (field-aware)
bm25f_results = search_bm25f(demo_query, top_k=TOP_K)
print("\n--- BM25F Field-Aware (title/abstract/claims/description) ---")
print(bm25f_results[["rank", "ucid", "bm25f_score", "score_title", "score_abstract", "score_claims", "title"]].to_string(index=False))

QUERY: 'wireless power transfer charging device'

--- BM25 Baseline (title+abstract only) ---
 rank             ucid  bm25_score                                                                                                                 title
    1 WO-2000002299-A1   17.219591                                                    IMPROVED POWER SUPPLY ASSEMBLY FOR HAND-HELD COMMUNICATIONS DEVICE
    2 WO-2000001192-A2   12.907973                                                               CALL ADMISSION CONTROL SYSTEM FOR WIRELESS ATM NETWORKS
    3 WO-2000001934-A1   12.632371                                                              OPERATING A GAS TURBINE WITH SUPPLEMENTAL COMPRESSED AIR
    4 WO-2000000928-A1   11.770257                                     APPARATUS AND METHODS FOR IMAGING WRITTEN INFORMATION WITH A MOBILE TELEPHONE SET
    5 WO-2000001132-A1   11.560131                                  TELEPHONE DIRECTORY MANAGEMENT SYSTEM HAVING WIRELESS TELEPHONE INTERFACE

### 7.1 — Patent-as-Query Sanity Check

In [11]:
demo_topic_ucid  = docs_df.iloc[0]["ucid"]
demo_topic_title = docs_df.iloc[0]["title"][:120]

print(f"Topic patent: {demo_topic_ucid}")
print(f"Title: {demo_topic_title}")
print()

query_text = f"{docs_df.iloc[0]['title']} {docs_df.iloc[0]['abstract']}".strip()

baseline_pat = search_bm25_baseline(query_text, top_k=TOP_K, exclude_ucid=demo_topic_ucid)
bm25f_pat    = search_bm25f(query_text, top_k=TOP_K, exclude_ucid=demo_topic_ucid)

print("--- BM25 Baseline ---")
print(baseline_pat[["rank", "ucid", "bm25_score", "title"]].to_string(index=False))
print("\n--- BM25F ---")
print(bm25f_pat[["rank", "ucid", "bm25f_score", "title"]].to_string(index=False))


Topic patent: WO-1979000001-A1
Title: APPARATUS FOR DETERMINING THE EPIDERMIC GROUP OF A LIVING BEING:DRY-PART DRY-PART GREASY-GREASY

--- BM25 Baseline ---
 rank             ucid  bm25_score                                                                                                                        title
    1 WO-2000001467-A1   29.988221                                                                      METHOD AND APPARATUS FOR PRODUCING HIGHLY CLEAN DRY AIR
    2 WO-2000000221-A1   28.624871                                                                                               WATER SOLUBLE DRY COMPOSITIONS
    3 WO-2000000452-A1   25.597825                                                                A PROCESS FOR THE PREPARATION OF COMPOUND FERTILIZER GRANULES
    4 WO-2000001775-A1   25.430142                                                                                                  POWDER COATING COMPOSITIONS
    5 WO-2000001397-A1   25.006263 WATER-SO

## 8 — Side-by-Side Comparison

**New in Step 2.** We merge the two ranked lists so you can see exactly which patents moved up or down, and by how much.

In [12]:
def compare_rankings(query_text: str, top_k: int = 10, exclude_ucid: str = None):
    """Runs BM25 and BM25F on the same query, merges results, and shows rank changes."""
    bm25_res  = search_bm25_baseline(query_text, top_k=top_k, exclude_ucid=exclude_ucid)
    bm25f_res = search_bm25f(query_text, top_k=top_k, exclude_ucid=exclude_ucid)

    all_ids = set(bm25_res["ucid"]).union(set(bm25f_res["ucid"]))
    rows = []
    for ucid in all_ids:
        bm25_rank  = bm25_res.loc[bm25_res["ucid"]   == ucid, "rank"]
        bm25f_rank = bm25f_res.loc[bm25f_res["ucid"] == ucid, "rank"]

        title = ""
        if not bm25_res.loc[bm25_res["ucid"] == ucid].empty:
            title = bm25_res.loc[bm25_res["ucid"] == ucid, "title"].iloc[0]
        elif not bm25f_res.loc[bm25f_res["ucid"] == ucid].empty:
            title = bm25f_res.loc[bm25f_res["ucid"] == ucid, "title"].iloc[0]

        r_bm25  = int(bm25_rank.iloc[0])  if not bm25_rank.empty  else None
        r_bm25f = int(bm25f_rank.iloc[0]) if not bm25f_rank.empty else None

        if r_bm25 is not None and r_bm25f is not None:
            change = r_bm25 - r_bm25f
        elif r_bm25f is not None:
            change = "NEW"
        else:
            change = "DROPPED"

        rows.append({
            "ucid": ucid, "title": title[:80],
            "rank_bm25": r_bm25 if r_bm25 is not None else "-",
            "rank_bm25f": r_bm25f if r_bm25f is not None else "-",
            "rank_change": change,
        })

    comp_df = pd.DataFrame(rows)
    comp_df["sort_key"] = comp_df["rank_bm25f"].apply(
        lambda x: int(x) if isinstance(x, (int, float)) else 999)
    return comp_df.sort_values("sort_key").drop(columns=["sort_key"]).reset_index(drop=True)


print(f"Query: '{demo_query}'")
comparison = compare_rankings(demo_query, top_k=TOP_K)
print(comparison.to_string(index=False))
print("\nrank_change > 0 = BM25F promoted | < 0 = demoted | NEW / DROPPED = entered/left top-k")


Query: 'wireless power transfer charging device'
            ucid                                                                            title rank_bm25 rank_bm25f rank_change
WO-2000002299-A1               IMPROVED POWER SUPPLY ASSEMBLY FOR HAND-HELD COMMUNICATIONS DEVICE         1          1           0
WO-2000001192-A2                          CALL ADMISSION CONTROL SYSTEM FOR WIRELESS ATM NETWORKS         2          2           0
WO-2000002358-A1                 SECURE SESSION SET UP BASED ON THE WIRELESS APPLICATION PROTOCOL         7          3           4
WO-2000000562-A1                                                         ADHESIVE TRANSFER DEVICE         -          4         NEW
WO-2000001132-A1 TELEPHONE DIRECTORY MANAGEMENT SYSTEM HAVING WIRELESS TELEPHONE INTERFACE CAPABI         5          5           0
WO-2000000928-A1 APPARATUS AND METHODS FOR IMAGING WRITTEN INFORMATION WITH A MOBILE TELEPHONE SE         4          6          -2
WO-2000000419-A1                  

The field coverage audit confirmed that all four patent fields (title, abstract, claims, description) have strong coverage (91–100%) across the 2,000-document subset, making BM25F a viable approach for this corpus.
The side-by-side comparison between plain BM25 and BM25F demonstrates that preserving document structure meaningfully changes retrieval rankings. BM25F promotes patents with consistent term matches across multiple fields (e.g., the wireless application protocol patent gaining +4 ranks due to strong claims evidence) and demotes patents that match only superficially in the title or abstract (e.g., the gas turbine patent dropping -6 ranks because its claims and description contain no relevant terms). BM25F also surfaces new results that plain BM25 misses entirely, such as the transcutaneous energy transfer device, which was discovered through claims and description matches alone.
However, both methods still retrieve some topically unrelated patents for broad queries, highlighting the vocabulary mismatch problem inherent in lexical retrieval. This motivates the next stage of the pipeline: dense semantic re-ranking to capture meaning beyond exact term overlap.

## 9 — Field Score Breakdown: Where Does the Signal Come From?

**New in Step 2.** For the top BM25F results, we show which field contributed most to the score. This is the key interpretability advantage of field-aware retrieval.

In [13]:
bm25f_detailed = search_bm25f(demo_query, top_k=TOP_K)

score_cols = ["score_title", "score_abstract", "score_claims", "score_description"]
display_cols = ["rank", "ucid", "bm25f_score"] + score_cols + ["title"]

print(f"Query: '{demo_query}'")
print(f"Weights: {FIELD_WEIGHTS}")
print()

# show which field dominates for each result
for _, row in bm25f_detailed.iterrows():
    raw_scores = {f: row[f"score_{f}"] for f in FIELDS}
    weighted_scores = {f: FIELD_WEIGHTS[f] * raw_scores[f] for f in FIELDS}
    dominant = max(weighted_scores, key=weighted_scores.get)
    print(f"Rank {row['rank']:>2} | {row['ucid']:>12} | BM25F={row['bm25f_score']:7.2f} | "
          f"dominant field: {dominant:>12} ({weighted_scores[dominant]:.2f}) | {row['title'][:60]}")

Query: 'wireless power transfer charging device'
Weights: {'title': 3.0, 'abstract': 2.0, 'claims': 1.5, 'description': 1.0}

Rank  1 | WO-2000002299-A1 | BM25F=  95.33 | dominant field:     abstract (31.70) | IMPROVED POWER SUPPLY ASSEMBLY FOR HAND-HELD COMMUNICATIONS 
Rank  2 | WO-2000001192-A2 | BM25F=  87.63 | dominant field:       claims (30.50) | CALL ADMISSION CONTROL SYSTEM FOR WIRELESS ATM NETWORKS
Rank  3 | WO-2000002358-A1 | BM25F=  69.45 | dominant field:     abstract (21.43) | SECURE SESSION SET UP BASED ON THE WIRELESS APPLICATION PROT
Rank  4 | WO-2000000562-A1 | BM25F=  68.67 | dominant field:        title (29.99) | ADHESIVE TRANSFER DEVICE
Rank  5 | WO-2000001132-A1 | BM25F=  66.93 | dominant field:     abstract (22.65) | TELEPHONE DIRECTORY MANAGEMENT SYSTEM HAVING WIRELESS TELEPH
Rank  6 | WO-2000000928-A1 | BM25F=  66.06 | dominant field:       claims (28.43) | APPARATUS AND METHODS FOR IMAGING WRITTEN INFORMATION WITH A
Rank  7 | WO-2000000419-A1 | BM25F=  64.05 | 

## 10 — Weight Sensitivity Analysis

**New in Step 2.** We try different weight configurations to see how sensitive rankings are to the choice of field weights.

In [14]:
weight_configs = {
    "equal":       {"title": 1.0, "abstract": 1.0, "claims": 1.0, "description": 1.0},
    "title_heavy":  {"title": 5.0, "abstract": 2.0, "claims": 1.0, "description": 0.5},
    "claims_heavy": {"title": 2.0, "abstract": 1.5, "claims": 4.0, "description": 1.0},
    "default":      FIELD_WEIGHTS,
}

print(f"Query: '{demo_query}'\n")

for config_name, weights in weight_configs.items():
    res = search_bm25f(demo_query, field_weights=weights, top_k=5)
    doc_list = res["ucid"].tolist()
    print(f"{config_name:>15}: {doc_list}")

print("\nIf the top-5 list changes across configs, field weighting matters for this query.")
print("If it stays the same, the ranking is robust to weight choices.")

Query: 'wireless power transfer charging device'

          equal: ['WO-2000002299-A1', 'WO-2000001192-A2', 'WO-2000000928-A1', 'WO-2000001934-A1', 'WO-2000002358-A1']
    title_heavy: ['WO-2000002299-A1', 'WO-2000001192-A2', 'WO-2000000562-A1', 'WO-2000000419-A1', 'WO-2000002358-A1']
   claims_heavy: ['WO-2000002299-A1', 'WO-2000001192-A2', 'WO-2000000928-A1', 'WO-2000001934-A1', 'WO-2000000847-A1']
        default: ['WO-2000002299-A1', 'WO-2000001192-A2', 'WO-2000002358-A1', 'WO-2000000562-A1', 'WO-2000001132-A1']

If the top-5 list changes across configs, field weighting matters for this query.
If it stays the same, the ranking is robust to weight choices.


Field weighting does matter for this query, especially for the middle ranks. The top results are robust, but the ordering below that depends on which field you trust most. This is a good finding for your presentation — it shows that BM25F isn't just a cosmetic change, the weight configuration is a genuine tuning decision.

## 11 — Find an Example Where BM25F Changes the Order

**New in Step 2.** We specifically look for a query where BM25F produces a meaningfully different top result or rank ordering compared to plain BM25, and explain why.

In [15]:
# try several queries and find one with the most rank movement
test_queries = [
    "wireless power transfer charging device",
    "semiconductor manufacturing process wafer",
    "optical fiber communication signal transmission",
    "combustion engine fuel injection control",
    "pharmaceutical composition treating cancer",
]

best_query = None
best_movement = 0
best_comparison = None

for q in test_queries:
    try:
        comp = compare_rankings(q, top_k=TOP_K)
        # count how many docs have a rank change (not zero)
        numeric_changes = comp["rank_change"].apply(
            lambda x: abs(int(x)) if isinstance(x, (int, float)) and x != 0
            else (1 if x in ["NEW", "DROPPED"] else 0)
        )
        total_movement = numeric_changes.sum()
        if total_movement > best_movement:
            best_movement = total_movement
            best_query = q
            best_comparison = comp
    except Exception as e:
        print(f"Skipping query '{q}': {e}")

if best_query:
    print(f"Query with most rank movement: '{best_query}'")
    print(f"Total rank displacement: {best_movement}")
    print()
    print(best_comparison.to_string(index=False))
else:
    print("No ranking differences found. Try more queries or a larger subset.")

Query with most rank movement: 'combustion engine fuel injection control'
Total rank displacement: 39

            ucid                                                                            title rank_bm25 rank_bm25f rank_change
WO-2000000725-A1                                   ENGINE SYSTEM EMPLOYING AN UNSYMMETRICAL CYCLE         8          1           7
WO-2000000732-A1                                                    FUEL SYSTEM FOR LIQUEFIED GAS         6          2           4
WO-1979000205-A1                                     A CARBURETOR FOR INTERNAL COMBUSTION ENGINES         9          3           6
WO-2000000739-A1 METHOD OF CONTROLLING THE IGNITION IN AN INTERNAL COMBUSTION ENGINE AND ENGINE W         -          4         NEW
WO-1979000053-A1                                   INTERNAL COMBUSTION ENGINE FUEL ECONOMY SYSTEM         5          5           0
WO-2000000770-A1                                             FUEL INJECTOR FOR GAS TURBINE ENGINE         -    

### 11.1 — Explain the Rank Change

For the best example above, show the per-field score breakdown for a patent that moved significantly.

In [16]:
if best_query:
    bm25f_detail = search_bm25f(best_query, top_k=TOP_K)

    print(f"Query: '{best_query}'")
    print(f"\nBM25F Top-{TOP_K} with per-field breakdown:")
    print()
    for _, row in bm25f_detail.iterrows():
        raw = {f: row[f"score_{f}"] for f in FIELDS}
        wtd = {f: FIELD_WEIGHTS[f] * raw[f] for f in FIELDS}
        contributions = " | ".join(f"{f}={wtd[f]:.2f}" for f in FIELDS)
        print(f"  Rank {row['rank']:>2}: {row['ucid']} (BM25F={row['bm25f_score']:.2f})")
        print(f"           {contributions}")
        print(f"           Title: {row['title'][:80]}")
        print()

Query: 'combustion engine fuel injection control'

BM25F Top-10 with per-field breakdown:

  Rank  1: WO-2000000725-A1 (BM25F=130.93)
           title=15.88 | abstract=33.19 | claims=54.23 | description=27.63
           Title: ENGINE SYSTEM EMPLOYING AN UNSYMMETRICAL CYCLE

  Rank  2: WO-2000000732-A1 (BM25F=118.35)
           title=15.77 | abstract=34.63 | claims=41.04 | description=26.90
           Title: FUEL SYSTEM FOR LIQUEFIED GAS

  Rank  3: WO-1979000205-A1 (BM25F=114.77)
           title=16.07 | abstract=25.36 | claims=46.76 | description=26.58
           Title: A CARBURETOR FOR INTERNAL COMBUSTION ENGINES

  Rank  4: WO-2000000739-A1 (BM25F=110.16)
           title=22.84 | abstract=27.00 | claims=33.54 | description=26.78
           Title: METHOD OF CONTROLLING THE IGNITION IN AN INTERNAL COMBUSTION ENGINE AND ENGINE W

  Rank  5: WO-1979000053-A1 (BM25F=109.73)
           title=46.73 | abstract=0.00 | claims=39.80 | description=23.19
           Title: INTERNAL COMBUSTION ENG

BM25F rewards documents with deep, consistent relevance across all fields, and correctly penalises documents that only match on the surface. The rank 10 example is the perfect case study, baseline's top result demoted to last place because it had zero evidence beyond title and abstract.

## 12 — Save Outputs

Save the BM25F ranked results and comparison tables for use in downstream notebooks (dense re-ranking, fusion).

In [17]:
# save BM25F demo results
bm25f_results_path = RESULTS_DIR / "bm25f_free_text_results.csv"
bm25f_detailed.to_csv(bm25f_results_path, index=False)

# save comparison table
if best_comparison is not None:
    comparison_path = RESULTS_DIR / "bm25_vs_bm25f_comparison.csv"
    best_comparison.to_csv(comparison_path, index=False)

# save field coverage audit
coverage_path = RESULTS_DIR / "field_coverage_audit.csv"
coverage_df.to_csv(coverage_path)

# save the weight config used
config_path = RESULTS_DIR / "bm25f_config.json"
config_path.write_text(json.dumps({
    "field_weights": FIELD_WEIGHTS,
    "top_k": TOP_K,
    "num_docs": len(docs_df),
}, indent=2))

print("Saved:")
for p in RESULTS_DIR.iterdir():
    print(f"  {p}")

Saved:
  /Users/ledionalame/Documents/QMUL/Second Semester/Information Retrieval/Assignment 2/Project/IR-CW2/results/bm25f/bm25f_free_text_results.csv
  /Users/ledionalame/Documents/QMUL/Second Semester/Information Retrieval/Assignment 2/Project/IR-CW2/results/bm25f/field_coverage_audit.csv
  /Users/ledionalame/Documents/QMUL/Second Semester/Information Retrieval/Assignment 2/Project/IR-CW2/results/bm25f/bm25_vs_bm25f_comparison.csv
  /Users/ledionalame/Documents/QMUL/Second Semester/Information Retrieval/Assignment 2/Project/IR-CW2/results/bm25f/bm25f_config.json
